# ***Starpower DevTools***

*This just a lil youtube converter for a model nothing crazy .. but can be helpful if your looking to develop an uncommon RAG system. I dont know about everyone else but i actually look for most information on houtube so it just seems reasonable for the model to do so also*

In [1]:
# Install the necessary libraries for API access, downloading, and AI transcription
!pip install -q google-api-python-client yt-dlp openai-whisper
# FFmpeg is usually pre-installed on Kaggle, but we ensure it's available
!apt-get install -y ffmpeg


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s

*For Youtube API Access*

If you need access to youtube you can go to this link below*

[https://console.cloud.google.com/apis/library/youtube.googleapis.com](http://)



In [2]:
import os
import sys
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import yt_dlp
import whisper
import torch



# Configuration
API_KEY = 'YOUTUBE_API_KEY'  # <--- PASTE YOUR API KEY HERE
YOUTUBE_API_SERVICE_NAME = 'youtube'
YOUTUBE_API_VERSION = 'v3'
WHISPER_MODEL_SIZE = 'medium' # Options: tiny, base, small, medium, large (requires GPU)

def search_youtube(query, max_results=5):
    """
    Step 2: Retrieve list of videos using the official YouTube Data API.
    Returns a list of dictionaries containing video title and ID.
    """
    try:
        youtube = build(YOUTUBE_API_SERVICE_NAME, YOUTUBE_API_VERSION, developerKey=API_KEY)
        
        # Call the search.list method to retrieve results matching the specified query term.
        search_response = youtube.search().list(
            q=query,
            type='video',
            part='id,snippet',
            maxResults=max_results
        ).execute()

        videos = []
        for search_result in search_response.get('items', []):
            videos.append({
                'title': search_result['snippet']['title'],
                'videoId': search_result['id']['videoId'],
                'url': f"https://www.youtube.com/watch?v={search_result['id']['videoId']}"
            })
        return videos

    except HttpError as e:
        print(f"An HTTP error {e.resp.status} occurred:\n{e.content}")
        return []

def download_audio_wav(video_url, output_name="target_audio"):
    """
    Step 4: Download audio and convert to WAV using yt-dlp and FFmpeg.
    This ensures compatibility with Whisper.
    """
    print(f"\n[System] Downloading and converting {video_url}...")
    
    # Configure yt-dlp for best audio quality converted to wav
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': output_name, # output filename pattern
        'quiet': True,
        'no_warnings': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_url])
        
        # Verify file creation (yt-dlp appends the extension)
        final_filename = f"{output_name}.wav"
        if os.path.exists(final_filename):
            print(f"[System] Audio saved to: {final_filename}")
            return final_filename
        else:
            raise FileNotFoundError("WAV file was not created successfully.")
            
    except Exception as e:
        print(f"[Error] Download failed: {e}")
        return None

def transcribe_audio(file_path):
    """
    Step 5: Convert WAV file into a transcript using OpenAI Whisper.
    """
    print(f"\n[System] Loading Whisper model ('{WHISPER_MODEL_SIZE}')...")
    
    # Check for GPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("[Warning] GPU not detected. Transcription will be slow.")
    
    try:
        model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)
        print(f"[System] Transcribing audio (this may take a moment)...")
        
        # Transcribe
        result = model.transcribe(file_path)
        return result['text']
        
    except Exception as e:
        print(f"[Error] Transcription failed: {e}")
        return None

# --- Main Execution Pipeline ---

if __name__ == "__main__":
    if API_KEY == 'YOUR_YOUTUBE_API_KEY_HERE':
        print("[Error] Please insert your Google Cloud API Key in the configuration section.")
    else:
        # Step 1: Type a query
        user_query = input("Step 1: Enter your search query: ")
        
        # Step 2: Retrieve list
        print(f"\nSearching for '{user_query}'...")
        results = search_youtube(user_query)
        
        if results:
            # Display options
            print(f"\n--- Found {len(results)} Videos ---")
            for idx, video in enumerate(results):
                print(f"{idx + 1}. {video['title']}")
            
            # Step 3: Select video
            try:
                selection = int(input("\nStep 3: Select a video number (1-5): ")) - 1
                if 0 <= selection < len(results):
                    selected_video = results[selection]
                    print(f"Selected: {selected_video['title']}")
                    
                    # Step 4: Convert to WAV
                    wav_file = download_audio_wav(selected_video['url'])
                    
                    if wav_file:
                        # Step 5: Whisper Transcription
                        transcript = transcribe_audio(wav_file)
                        
                        # End Results
                        print("\n" + "="*40)
                        print("FINAL TRANSCRIPTION")
                        print("="*40)
                        print(transcript.strip())
                        print("="*40)
                        
                        # Cleanup (Optional)
                        # os.remove(wav_file)
                else:
                    print("[Error] Invalid selection.")
            except ValueError:
                print("[Error] Please enter a valid number.")
        else:
            print("[System] No videos found. Check your API quota or query.")


StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

In [ ]:
# Cell 1 — Install deps (Kaggle) @v1.0
# NOTE: If Kaggle already has some of these, pip will no-op quickly.

!pip -q install -U "transformers>=4.45.0" "accelerate>=0.33.0" "safetensors>=0.4.3"
!pip -q install -U "google-api-python-client>=2.140.0" "yt-dlp>=2024.8.6"
!pip -q install -U "openai-whisper>=20240930"

In [ ]:
# Cell 2 — Config + Secrets @v1.0

import os
import torch

# --- YouTube API Key (set this in Kaggle: Add-ons -> Secrets) ---
# Name it: YOUTUBE_API_KEY
YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY", "").strip()

# --- Model path (local or HF) ---
# If you have Qwen weights in /kaggle/input, set QWEN_MODEL_PATH to that folder.
# Otherwise it will try HF ID (requires internet enabled).
QWEN_MODEL_PATH = os.environ.get("QWEN_MODEL_PATH", "").strip()
QWEN_HF_ID = "Qwen/Qwen2.5-3B-Instruct"

# --- Whisper config ---
WHISPER_MODEL_SIZE = os.environ.get("WHISPER_MODEL_SIZE", "medium").strip()  # tiny/base/small/medium/large
AUDIO_OUT_NAME = "target_audio"  # will produce target_audio.wav

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"[config] DEVICE={DEVICE} DTYPE={DTYPE}")
print(f"[config] QWEN_MODEL_PATH={'(env)' if QWEN_MODEL_PATH else '(none)'} -> {QWEN_MODEL_PATH or QWEN_HF_ID}")
print(f"[config] WHISPER_MODEL_SIZE={WHISPER_MODEL_SIZE}")
print(f"[config] YOUTUBE_API_KEY={'set' if YOUTUBE_API_KEY else 'NOT SET'}")

In [ ]:
# Cell 3 — YouTube + Whisper Tools @v1.0

import json
from typing import Any, Dict, List, Optional
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import yt_dlp
import whisper

YOUTUBE_API_SERVICE_NAME = "youtube"
YOUTUBE_API_VERSION = "v3"

def tool_search_youtube(query: str, max_results: int = 5) -> List[Dict[str, str]]:
    """
    Search YouTube via Data API.
    Returns list[{title, videoId, url}]
    """
    if not YOUTUBE_API_KEY:
        raise RuntimeError("YOUTUBE_API_KEY is not set (Kaggle Secrets -> add YOUTUBE_API_KEY).")

    youtube = build(YOUTUBE_API_SERVICE_NAME, YOUTUBE_API_VERSION, developerKey=YOUTUBE_API_KEY)

    try:
        resp = youtube.search().list(
            q=query,
            type="video",
            part="id,snippet",
            maxResults=max_results
        ).execute()

        out: List[Dict[str, str]] = []
        for item in resp.get("items", []):
            vid = item["id"]["videoId"]
            out.append({
                "title": item["snippet"]["title"],
                "videoId": vid,
                "url": f"https://www.youtube.com/watch?v={vid}",
            })
        return out

    except HttpError as e:
        raise RuntimeError(f"YouTube API HttpError {getattr(e.resp,'status',None)}: {getattr(e,'content',e)}") from e


def tool_download_audio_wav(video_url: str, output_name: str = AUDIO_OUT_NAME) -> str:
    """
    Download best audio, convert to WAV (requires ffmpeg available).
    Returns path to wav file.
    """
    final_filename = f"{output_name}.wav"
    if os.path.exists(final_filename):
        os.remove(final_filename)

    ydl_opts = {
        "format": "bestaudio/best",
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "wav",
            "preferredquality": "192",
        }],
        # Ensure predictable name; yt-dlp will add .wav after postprocess
        "outtmpl": output_name,
        "quiet": True,
        "no_warnings": True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_url])
    except Exception as e:
        raise RuntimeError(f"yt-dlp download/convert failed: {e}") from e

    if not os.path.exists(final_filename):
        raise RuntimeError("WAV was not created. (ffmpeg missing or yt-dlp postprocess failed)")

    return final_filename


def tool_transcribe_wav(wav_path: str) -> str:
    """
    Whisper transcription. Returns transcript text (no timestamps).
    """
    if not os.path.exists(wav_path):
        raise RuntimeError(f"Audio file not found: {wav_path}")

    model = whisper.load_model(WHISPER_MODEL_SIZE, device=DEVICE)
    result = model.transcribe(wav_path)
    text = (result.get("text") or "").strip()

    if not text:
        raise RuntimeError("Whisper returned empty transcript.")
    return text


def tool_transcribe_first_from_query(query: str, max_results: int = 5) -> Dict[str, Any]:
    """
    Search videos -> pick first result -> download -> transcribe -> return bundle.
    Explicit: if any step fails, it raises with the real reason.
    """
    vids = tool_search_youtube(query=query, max_results=max_results)
    if not vids:
        raise RuntimeError("No videos found for query.")

    chosen = vids[0]
    wav = tool_download_audio_wav(chosen["url"], output_name=AUDIO_OUT_NAME)
    transcript = tool_transcribe_wav(wav)

    return {
        "query": query,
        "video": chosen,
        "wav_path": wav,
        "transcript": transcript,
    }

# ***Load Your Model***

*Im choosing qwen 2.5 3b because its free & easy to install .. doesnt take a long time .. but u can connect any model u just need to make sure that model is smart enough to handle a function calling tool intuitively for the autonomous setup*

In [ ]:
# Cell 4 — Load Qwen 2.5 3B Instruct @v1.0

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = QWEN_MODEL_PATH if QWEN_MODEL_PATH else QWEN_HF_ID

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True
)

model.eval()
print("[model] loaded:", MODEL_ID)

In [ ]:
# Cell 3.9 — Install YouTube + Whisper deps @v1.0

import sys, subprocess

def _pip(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# YouTube download + audio conversion + transcription
_pip("yt-dlp")
_pip("ffmpeg-python")
_pip("faster-whisper")

print("[deps] installed: yt-dlp, ffmpeg-python, faster-whisper")

In [ ]:
# Cell 4 — Load Qwen 2.5 3B Instruct @v1.1

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- required vars (must exist from earlier cells) ----
_required = ["QWEN_MODEL_PATH", "QWEN_HF_ID", "DTYPE", "DEVICE"]
_missing = [k for k in _required if k not in globals()]
if _missing:
    raise NameError(f"Missing required vars from earlier cells: {_missing}")

MODEL_ID = QWEN_MODEL_PATH if QWEN_MODEL_PATH else QWEN_HF_ID

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True
)

model.eval()
print("[model] loaded:", MODEL_ID)

In [ ]:
# Cell 4.1 — YouTube -> WAV (download + convert) @v1.0

import os, re, shutil, tempfile
import ffmpeg
from yt_dlp import YoutubeDL

_YT_RE = re.compile(r"(https?://)?(www\.)?(youtube\.com/watch\?v=|youtu\.be/)[\w\-]+")

def _is_youtube_link(text: str) -> bool:
    return bool(_YT_RE.search(text.strip()))

def youtube_to_wav16k_mono(youtube_url: str) -> str:
    """
    Downloads best audio from YouTube to a temp folder, converts to 16kHz mono WAV.
    Returns the WAV filepath.
    """
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg binary not found on this runtime. Install ffmpeg (system) and retry.")

    tmpdir = tempfile.mkdtemp(prefix="yt_audio_")
    outtmpl = os.path.join(tmpdir, "audio.%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": outtmpl,
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True,
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        downloaded_path = ydl.prepare_filename(info)

    wav_path = os.path.join(tmpdir, "audio.wav")

    (
        ffmpeg
        .input(downloaded_path)
        .output(wav_path, ac=1, ar=16000, format="wav")
        .overwrite_output()
        .run(quiet=True)
    )

    if not os.path.exists(wav_path) or os.path.getsize(wav_path) == 0:
        raise RuntimeError("WAV conversion failed (empty output).")

    return wav_path

In [ ]:
# Cell 4.2 — Transcribe WAV (faster-whisper) @v1.0

from faster_whisper import WhisperModel

# You can tune this if you want: "tiny", "base", "small", "medium", "large-v3"
WHISPER_SIZE = "small"

# Compute type choices:
# - CUDA: "float16" is typical
# - CPU: "int8" is typical
_compute_type = "float16" if DEVICE == "cuda" else "int8"

whisper_model = WhisperModel(
    WHISPER_SIZE,
    device="cuda" if DEVICE == "cuda" else "cpu",
    compute_type=_compute_type
)

def transcribe_wav(wav_path: str) -> str:
    """
    Returns full transcript text.
    """
    segments, info = whisper_model.transcribe(wav_path, beam_size=5, vad_filter=True)
    parts = []
    for seg in segments:
        if seg.text:
            parts.append(seg.text.strip())
    return " ".join(parts).strip()

In [ ]:
# Cell 4.3 — Router: YouTube link -> transcript -> Qwen response @v1.0

def build_model_input(user_message: str) -> str:
    user_message = user_message.strip()

    if _is_youtube_link(user_message):
        wav_path = youtube_to_wav16k_mono(user_message)
        transcript = transcribe_wav(wav_path)

        if not transcript:
            raise RuntimeError("Transcription returned empty text.")

        return (
            f"User message:\n{user_message}\n\n"
            f"[TRANSCRIPT]\n{transcript}\n"
        )

    return f"User message:\n{user_message}\n"

@torch.no_grad()
def qwen_generate(prompt: str, max_new_tokens: int = 384, temperature: float = 0.7, top_p: float = 0.9) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        eos_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    # Return only the newly generated portion if you want, but leaving full text is safer for now.
    return text

def chat(user_message: str) -> str:
    prompt = build_model_input(user_message)
    return qwen_generate(prompt)

print("[router] ready: chat('youtube link OR normal text')")

# *Extras*

In [ ]:
# How it works

"""

if a use sends a video with a youtube link the system has to transcribe it using the cells above

user message: "youtube link" -> convert to wav -> transcribe audio

once the audio is transcribed, the transcript needs to be sent to the model like shown below

input = (user message + transcript)

this way recieving a video will register to the model as recieving a transcript with the user input so it can respond naturally)


"""

In [ ]:
# Cell 5 — The Starpower "Video Interceptor" Loop @v1.0
import re

def generate_response(user_input, transcript=None):
    """
    Constructs the prompt and gets a response from Qwen.
    If a transcript exists, it injects it into the context transparently.
    """
    
    # 1. Build the System Context
    if transcript:
        system_msg = (
            "You are an intelligent assistant. The user has provided a video link. "
            "Below is the TRANSCRIPT of that video. Use it to answer the user's question accurately.\n\n"
            f"--- VIDEO TRANSCRIPT START ---\n{transcript}\n--- VIDEO TRANSCRIPT END ---\n"
        )
    else:
        system_msg = "You are a helpful assistant."

    # 2. Format for Qwen 2.5 (Chat Template)
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_input}
    ]
    
    # 3. Tokenize & Generate
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(DEVICE)

    # (Print a little status so you know it's thinking)
    print("\n[🧠 Model] Reading and thinking...")
    
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True
    )
    
    # 4. Decode
    # We slice off the input tokens to just get the new response
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # Clean up (remove the prompt part if the model echoes it, though skip_special usually handles it)
    # For Qwen, usually we just want the last part, but 'response' here might contain the full convo depending on the tokenizer config.
    # A safe bet is to look for "assistant" or just print the whole thing if unsure. 
    # Let's try to extract just the answer.
    if "assistant\n" in response:
        return response.split("assistant\n")[-1]
    return response

# --- MAIN INTERACTION LOOP ---
print("==========================================")
print("   STARPOWER VIDEO AGENT ONLINE 🚀")
print("   Paste a YouTube link to have the model 'watch' it.")
print("   Type 'exit' to quit.")
print("==========================================\n")

while True:
    user_msg = input("You: ")
    
    if user_msg.lower() in ["exit", "quit"]:
        print("Starpower going offline.")
        break

    # 1. SCAN: Look for YouTube Link
    # Regex catches 'youtube.com' or 'youtu.be'
    youtube_regex = r'(https?://(?:www\.)?(?:youtube\.com/watch\?v=|youtu\.be/)([\w-]+))'
    match = re.search(youtube_regex, user_msg)
    
    final_transcript = None

    if match:
        video_url = match.group(0)
        print(f"\n[👀 System] Detected Video Link: {video_url}")
        print("[⏳ System] Downloading and Transcribing... (This might take a sec)")
        
        try:
            # A. Download
            wav_path = tool_download_audio_wav(video_url)
            
            # B. Transcribe
            # Note: We use your tool_transcribe_wav from Cell 3
            final_transcript = tool_transcribe_wav(wav_path)
            
            print(f"[✅ System] Transcription Success! ({len(final_transcript)} chars)")
            
        except Exception as e:
            print(f"[❌ Error] Failed to process video: {e}")
            # We continue anyway, just without the transcript
    
    # 2. GENERATE: Send to Model
    response = generate_response(user_msg, final_transcript)
    
    print(f"\nStarpower: {response}\n")
    print("-" * 40)
